In [1]:
from kafka import KafkaConsumer
import time
from torchvision import datasets, transforms
from PIL import Image
import shutil
import os
import random
import subprocess
import logging
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)
from datetime import datetime
import json
import matplotlib.pyplot as plt
from neo4j import GraphDatabase, Session
import networkx as nx
import uuid

logger = logging.getLogger(__name__)
__file__ = "training.ipynb"

KAFKA_BOOTSTRAP_SERVERS = "localhost:29092"
KAFKA_TOPICS = [
    "connector-output-topic",
    "line-detector-output-topic",
    "angle-point-detector-output-topic",
    "skeletonization-output-topic",
    "contour-analysis-output-topic",
    "classification-output-topic",
    "dlq-topic",
]
KAFKA_CONSUMER_TIMEOUT_MS = 1000
KAFKA_POLL_TIMEOUT_MS = 1000
CLASSIFICATION_TIMEOUT_SECS = 120
CONFUSION_MATRIX_FIGSIZE = (10, 7)
BINARIES_PATH = os.path.join(os.path.dirname(__file__), os.pardir, "models", "binaries")
TRAINING_RESULTS_DIR = "training_results"
uri = "bolt://localhost:7687"
user = "neo4j"
password = "111122223333"

In [2]:
def generate_mnist_samples(
    number: int,
    max_samples: int = 100,
    test_fraction: float = 0.2,
    output_dir: str = "../../tests/generated_samples",
    randomize: bool = True,
) -> None:
    """
    Generate and save MNIST samples for a specified number, split into train and test sets.

    Args:
        number (int): The MNIST digit to generate samples for (0-9).
        max_samples (int, optional): The maximum number of samples to generate. Defaults to 100.
        test_fraction (float, optional): Fraction of samples to use for test set. Defaults to 0.2.
        output_dir (str, optional): The base output directory. Defaults to "../../tests/generated_samples".

    Returns:
        None
    """
    # Set up the output directories
    digit_output_dir = os.path.join(output_dir, f"mnist_{number}")
    train_dir = os.path.join(digit_output_dir, "train")
    test_dir = os.path.join(digit_output_dir, "test")
    # Remove existing directories with files inside
    if os.path.exists(digit_output_dir):
        shutil.rmtree(digit_output_dir)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Download and load MNIST dataset
    transform = transforms.Compose([transforms.ToTensor()])
    mnist_train = datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )

    # Filter for only the specified number
    filtered_dataset = [(img, label) for img, label in mnist_train if label == number]
    filtered_dataset = (
        filtered_dataset[:max_samples]
        if not randomize
        else random.sample(filtered_dataset, min(max_samples, len(filtered_dataset)))
    )

    # Calculate split indices
    test_size = int(len(filtered_dataset) * test_fraction)
    train_size = len(filtered_dataset) - test_size

    # Split dataset into train and test
    train_dataset = filtered_dataset[:train_size]
    test_dataset = filtered_dataset[train_size:]

    # Generate and save training images
    for i, (img, _) in enumerate(train_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(train_dir, f"mnist_{number}_{i:05d}.png"))

    # Generate and save test images with separate counter
    for i, (img, _) in enumerate(test_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(test_dir, f"mnist_{number}_{i:05d}.png"))

    print(
        f"Generated {train_size} training images and {test_size} test images of the number {number}"
    )
    print(f"Training images in: {train_dir}")
    print(f"Test images in: {test_dir}")

In [3]:
class Neo4jToNetworkX:
    """Responsible for converting Neo4j graphs to NetworkX format"""

    @staticmethod
    def extract_composed_graph_of_class(session: Session, class_name: str) -> nx.Graph:
        query = """
        MATCH (n)
        WHERE n.session_id = $class_name AND (n:Point OR n:Vector)
        WITH n, labels(n) as node_labels, properties(n) as node_props
        OPTIONAL MATCH (n)-[r]-(m)
        WHERE m.session_id = $class_name AND (m:Point OR m:Vector)
        WITH n, node_labels, node_props, r, m
        RETURN elementId(n) as node_id,
               node_labels,
               node_props,
               type(r) as rel_type,
               elementId(m) as target_id
        """
        result = session.run(query, class_name=class_name)
        return Neo4jToNetworkX._build_networkx_graph(result)

    @staticmethod
    def extract_image_graph(session: Session, image_id: str) -> nx.Graph:
        """
        Extracts a graph from Neo4j for a given image_id and converts it to NetworkX format.
        This version excludes segment embedding (i.e. no Segment nodes or HAS_RELATIVE_POSITION edges).
        """
        query = """
        MATCH (n {image_id: $image_id})
        WHERE n:Point OR n:Vector
        WITH n, labels(n) as node_labels, properties(n) as node_props
        OPTIONAL MATCH (n)-[r]-(m {image_id: $image_id})
        WITH n, node_labels, r, m, node_props
        RETURN elementId(n) AS node_id, 
               node_labels,
               node_props,
               type(r) AS rel_type, 
               elementId(m) AS target_id
        """
        result = session.run(query, image_id=image_id)
        return Neo4jToNetworkX._build_networkx_graph(result)

    @staticmethod
    def _build_networkx_graph(result) -> nx.Graph:
        G = nx.Graph()
        nodes: Dict[int, Dict] = {}

        records = list(result)
        for record in records:
            node_id = record["node_id"]
            if node_id not in nodes:
                node_data = {
                    "labels": set(record["node_labels"]),
                    **record["node_props"],
                }
                nodes[node_id] = node_data

        for node_id, node_data in nodes.items():
            G.add_node(node_id, **node_data)

        for record in records:
            if record["target_id"] is not None:
                G.add_edge(
                    record["node_id"], record["target_id"], type=record["rel_type"]
                )

        return G

In [4]:
def wait_for_kafka_idle(
    topic: str, idle_timeout: int = 30, bootstrap_servers: str = "localhost:29092"
) -> None:
    """
    Wait until a Kafka topic has been idle (no new messages) for the specified duration.

    Args:
        topic (str): Name of the Kafka topic to monitor
        idle_timeout (int, optional): Time in seconds to wait for no activity before considering idle. Defaults to 30.
        bootstrap_servers (str, optional): Kafka bootstrap servers. Defaults to "localhost:29092".

    Returns:
        None
    """

    # Create consumer
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=bootstrap_servers,
        auto_offset_reset="latest",
        enable_auto_commit=True,
        group_id=None,
        consumer_timeout_ms=1000,  # 1 second timeout for poll()
    )

    try:
        last_message_time = time.time()
        print(f"Monitoring topic {topic} for {idle_timeout} seconds of inactivity...")

        while True:
            # Try to get message
            messages = consumer.poll(timeout_ms=1000)
            current_time = time.time()

            if messages:
                # Reset timer if we got messages
                last_message_time = current_time
                print("Messages received, resetting idle timer...")
            else:
                # Check if we've been idle long enough
                idle_duration = current_time - last_message_time
                if idle_duration >= idle_timeout:
                    print(
                        f"No messages received for {idle_timeout} seconds. Topic {topic} is idle."
                    )
                    return

                if idle_duration >= 5:  # Only print every 5 seconds
                    print(f"No messages for {int(idle_duration)} seconds...")

    finally:
        consumer.close()

In [5]:
def train_mnist(
    class_number: int,
    subclass: int | None = None,
    samples: int | None = None,
    is_prepared_samples: bool = False,
    with_post_process: bool = True,
) -> None:
    """
    Train MNIST classifier for a specific class and optional subclass.

    Args:
        class_number (int): The main class number
        subclass (int | None): Optional subclass number
        samples (int | None): Number of samples to use
        is_prepared_samples (bool): Whether to use prepared samples
    """
    concept_id = str(class_number) + (f"_{subclass}" if subclass is not None else "")

    if is_prepared_samples:
        if subclass is not None:
            subprocess.run(
                ["make", f"train_prepared_samples_{class_number}", str(subclass)],
                cwd="../../",
            )
        else:
            subprocess.run(
                ["make", f"train_prepared_samples_{class_number}"], cwd="../../"
            )
    else:
        generate_mnist_samples(class_number, max_samples=samples)
        subprocess.run(["make", f"train_mnist_{class_number}"], cwd="../../")

    wait_for_kafka_idle(
        topic="contour-analysis-output-topic",
        idle_timeout=10,
        bootstrap_servers="localhost:29092",
    )
    if with_post_process:
        subprocess.run(
            [
                "make",
                "post_process",
                concept_id,
                f"mnist-{class_number}",
            ],
            cwd="../../",
        )

    if not verify_concept_created(concept_id):
        raise RuntimeError(
            f"Concept creation failed: concept '{concept_id}' was not found in Neo4j after training. "
            f"Check the pipeline logs for errors."
        )

In [6]:
from neo4j import GraphDatabase


def clean_neo4j_db() -> None:
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Delete nodes in batches of 10000 to avoid memory issues
        while True:
            result = session.run(
                """
                MATCH (n) 
                WITH n LIMIT 10000
                DETACH DELETE n
                RETURN count(n) as deleted
            """
            )
            if result.single()["deleted"] == 0:
                break
    driver.close()


def delete_test_neo4j_nodes() -> None:
    """Deletes all test nodes from Neo4j database"""
    uri = "bolt://localhost:7687"
    user = "neo4j"
    password = "111122223333"

    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Delete all nodes and relationships
        session.run("MATCH (n {session_id: 'test'}) DETACH DELETE n")
    driver.close()


def clean_kafka_topics() -> None:
    """Deletes all messages from Kafka topics by recreating them"""
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError, UnknownTopicOrPartitionError

    topics = [
        "connector-output-topic",
        "line-detector-output-topic",
        "angle-point-detector-output-topic",
        "skeletonization-output-topic",
        "contour-analysis-output-topic",
        "classification-output-topic",
        "dlq-topic",
    ]

    admin_client = KafkaAdminClient(bootstrap_servers="localhost:29092")

    # Delete existing topics
    for topic in topics:
        try:
            admin_client.delete_topics([topic])
            logging.info(f"Deleted topic: {topic}")
        except UnknownTopicOrPartitionError:
            logging.info(f"Topic {topic} does not exist")

    time.sleep(5)  # Wait for topics to be fully deleted

    # Recreate topics
    topic_list = []
    for topic in topics:
        topic_list.append(NewTopic(name=topic, num_partitions=1, replication_factor=1))

    for topic in topic_list:
        try:
            admin_client.create_topics([topic])
            logging.info(f"Created topic: {topic.name}")
        except TopicAlreadyExistsError:
            logging.warning(f"Topic {topic.name} already exists")

    admin_client.close()

In [7]:
def verify_concept_created(concept_id: str) -> bool:
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        result = session.run(
            "MATCH (n {concept_id: $concept_id}) RETURN count(n) as count",
            concept_id=concept_id,
        )
        count = result.single()["count"]
    driver.close()
    return count > 0

In [8]:
def classify_image(
    image_path: str,
    expected_name: str | None = None,
    timeout: int = CLASSIFICATION_TIMEOUT_SECS,
    params: Dict[str, Any] | None = None,
) -> Dict[str, Any]:
    """
    Classify a single image and get results from Kafka.

    Args:
        image_path: Path to the image file
        expected_name: Optional expected concept name for validation
        timeout: Timeout in seconds for waiting for classification result

    Returns:
        Dictionary containing classification results or error information
    """
    consumer = KafkaConsumer(
        "classification-output-topic",
        "dlq-topic",
        bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        enable_auto_commit=True,
        value_deserializer=lambda x: json.loads(x.decode("utf-8")),
        group_id=f"classify-single-{int(time.time())}",  # Unique group ID
        consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS,
    )

    try:
        # Create base parameters
        parameters = {"image_path": image_path}
        if params:
            parameters.update(params)

        # Create properly formatted JSON string
        params_json = json.dumps(parameters)  # Convert to JSON string
        params_escaped = params_json.replace('"', '\\"')  # Escape quotes for shell

        subprocess.run(
            ["make", "classify", f"PARAMS={params_escaped}"],
            cwd="../../",
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        start_time = time.time()
        result: Dict[str, Any] = {"status": "unknown"}

        while time.time() - start_time < timeout:
            try:
                messages = consumer.poll(timeout_ms=KAFKA_POLL_TIMEOUT_MS)
                for topic_partition, msgs in messages.items():
                    for msg in msgs:
                        if (
                            msg.topic == "dlq-topic"
                            and msg.value["value"]["parameters"]["image_id"]
                            == parameters["image_id"]
                        ):
                            result = {
                                "status": "error",
                                "image_id": msg.value["value"]["parameters"][
                                    "image_id"
                                ],
                                "image_path": image_path,
                                "error": "DLQ",
                            }
                            if expected_name:
                                result["expected"] = expected_name
                            return result

                        elif msg.topic == "classification-output-topic":
                            kafka_result = msg.value

                            # Verify the result corresponds to the current image
                            if kafka_result["image_id"] != parameters["image_id"]:
                                continue

                            result = {
                                "status": "success",
                                "image_path": image_path,
                                **kafka_result,
                            }

                            if expected_name:
                                result["expected"] = expected_name
                                if (
                                    "classification_results" in kafka_result
                                    and kafka_result["classification_results"].__len__()
                                    > 0
                                ):
                                    predicted_class = kafka_result[
                                        "classification_results"
                                    ][0]["concept_id"].split("_")[0]
                                    result["predicted"] = predicted_class
                                    result["correct"] = predicted_class == expected_name
                                else:
                                    result["correct"] = False
                                    result["predicted"] = "not classified"

                            return result

            except Exception as e:
                logger.error(f"Error reading from Kafka: {e}")
                time.sleep(0.1)
                continue

        # Timeout case
        result = {
            "status": "timeout",
            "image_id": parameters["image_id"],
            "image_path": image_path,
            "error": "Classification timeout",
        }
        if expected_name:
            result["expected"] = expected_name

        return result

    finally:
        consumer.close()

In [9]:
def save_confusion_matrix(cm: np.ndarray, classes: List[str], run_dir: str) -> None:
    """Plot and save confusion matrix."""
    plt.figure(figsize=CONFUSION_MATRIX_FIGSIZE)
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Add text annotations
    thresh = cm.max() / 2.0
    for i, j in np.ndindex(cm.shape):
        plt.text(
            j,
            i,
            format(cm[i, j], "d"),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()

    # Save plot in run directory
    plt.savefig(os.path.join(run_dir, "confusion_matrix.png"))
    plt.close()

In [10]:
from tqdm.notebook import tqdm


def test_mnist_all(
    classes: List[int], params: Dict[str, Any], sample_fraction: float = 1.0
) -> Tuple[Dict[str, Any], List[str], List[str]]:
    """Test MNIST classification for all classes and calculate overall metrics.

    Args:
        classes: List of class numbers to test
        sample_fraction: Fraction of test images to use per class (0.0-1.0). Default 1.0 (all).

    Returns:
        Tuple containing results dict, true labels and predicted labels
    """
    all_results: Dict[str, Any] = {}
    all_y_true: List[str] = []
    all_y_pred: List[str] = []
    incorrect_results: List[Dict[str, Any]] = []

    # Create run directory with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(TRAINING_RESULTS_DIR, f"run_{timestamp}")
    os.makedirs(run_dir, exist_ok=True)

    # Collect test images per class, applying sample_fraction
    class_images: Dict[int, List[str]] = {}
    total_images = 0
    for class_number in classes:
        test_folder = f"../../tests/generated_samples/mnist_{class_number}/test"
        images = [
            f for f in os.listdir(test_folder) if f.endswith((".png", ".jpg", ".jpeg"))
        ]
        if sample_fraction < 1.0:
            k = max(1, int(len(images) * sample_fraction))
            images = random.sample(images, k)
        class_images[class_number] = images
        total_images += len(images)

    if sample_fraction < 1.0:
        print(f"Using {sample_fraction:.0%} sample → {total_images} images total")

    # Initialize progress bar
    pbar = tqdm(total=total_images, desc="Testing MNIST classification", position=0, leave=True)

    for class_number in classes:
        pbar.set_description(f"Testing class {class_number}")
        nuclio_volume_path = (
            f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"
        )
        test_images = [os.path.join(nuclio_volume_path, f) for f in class_images[class_number]]
        expected_name = str(class_number)

        for image_file in test_images:
            params["image_id"] = str(uuid.uuid4())
            result = classify_image(
                image_file, expected_name=expected_name, params=params
            )

            all_results[image_file] = result

            if result["status"] == "success":
                all_y_true.append(result["expected"])
                all_y_pred.append(result["predicted"])

                if not result["correct"]:
                    incorrect_results.append(result)
            else:
                incorrect_results.append(result)

            pbar.update(1)

    pbar.close()

    # Calculate overall metrics
    failed_dlq = sum(1 for r in incorrect_results if r.get("error") == "DLQ")
    successful = len(all_y_true)

    # Save incorrect results to CSV
    if incorrect_results:
        incorrect_df = pd.DataFrame(incorrect_results)
        incorrect_df.to_csv(os.path.join(run_dir, "incorrect_results.csv"), index=False)

    if successful > 0:
        # Calculate metrics and save results
        labels = sorted(list(set(all_y_true + all_y_pred)))

        # Calculate overall metrics
        overall_precision, overall_recall, overall_f1, _ = (
            precision_recall_fscore_support(
                all_y_true, all_y_pred, labels=labels, average="weighted"
            )
        )
        overall_accuracy = accuracy_score(all_y_true, all_y_pred)

        # Calculate per-class metrics
        class_precision, class_recall, class_f1, support = (
            precision_recall_fscore_support(
                all_y_true, all_y_pred, labels=labels, average=None
            )
        )

        # Save metrics to CSV files and log results
        _save_metrics(
            run_dir,
            total_images,
            failed_dlq,
            successful,
            overall_accuracy,
            overall_precision,
            overall_recall,
            overall_f1,
            labels,
            class_precision,
            class_recall,
            class_f1,
            support,
        )

        # Generate and save confusion matrix
        cm = confusion_matrix(all_y_true, all_y_pred, labels=labels)
        save_confusion_matrix(cm, labels, run_dir)
    else:
        logger.warning("No successful classifications to calculate metrics")

    return all_results, all_y_true, all_y_pred


def _save_metrics(
    run_dir: str,
    total: int,
    failed_dlq: int,
    successful: int,
    overall_accuracy: float,
    overall_precision: float,
    overall_recall: float,
    overall_f1: float,
    labels: List[str],
    class_precision: np.ndarray,
    class_recall: np.ndarray,
    class_f1: np.ndarray,
    support: np.ndarray,
) -> None:
    """Save metrics to CSV files and log results."""
    # Save overall metrics
    metrics_df = pd.DataFrame(
        {
            "Metric": [
                "Total Images",
                "Failed (DLQ)",
                "Successfully Classified",
                "Success Rate (%)",
                "Accuracy (%)",
                "Precision (%)",
                "Recall (%)",
                "F1 Score (%)",
            ],
            "Value": [
                total,
                failed_dlq,
                successful,
                (successful / (total - failed_dlq)) * 100,
                overall_accuracy * 100,
                overall_precision * 100,
                overall_recall * 100,
                overall_f1 * 100,
            ],
        }
    )
    metrics_df.to_csv(os.path.join(run_dir, "metrics.csv"), index=False)

    # Save per-class metrics
    per_class_data = []
    for i, label in enumerate(labels):
        if label not in ["error", "timeout"]:
            per_class_data.append(
                {
                    "class": label,
                    "precision": class_precision[i] * 100,
                    "recall": class_recall[i] * 100,
                    "f1_score": class_f1[i] * 100,
                    "support": support[i],
                }
            )
    per_class_df = pd.DataFrame(per_class_data)
    per_class_df.to_csv(os.path.join(run_dir, "per_class_metrics.csv"), index=False)

    # Log results
    logger.info("\nOverall Classification Metrics:")
    logger.info(f"Total images across all classes: {total}")
    logger.info(f"Failed (DLQ): {failed_dlq}")
    logger.info(f"Successfully classified: {successful}")
    logger.info(f"Overall success rate: {(successful/(total-failed_dlq))*100:.2f}%")
    logger.info(f"Overall accuracy: {overall_accuracy*100:.2f}%")
    logger.info(f"Overall precision: {overall_precision*100:.2f}%")
    logger.info(f"Overall recall: {overall_recall*100:.2f}%")
    logger.info(f"Overall F1 Score: {overall_f1*100:.2f}%")

    logger.info("\nPer-Class Metrics:")
    for i, label in enumerate(labels):
        if label not in ["error", "timeout"]:
            logger.info(f"\n{label}:")
            logger.info(f"Precision: {class_precision[i]*100:.2f}%")
            logger.info(f"Recall: {class_recall[i]*100:.2f}%")
            logger.info(f"F1 Score: {class_f1[i]*100:.2f}%")
            logger.info(f"Support: {support[i]}")

In [11]:
# classes_to_subclasses = {
#     # 0: [1],
#     1: [1, 2, 3],
#     2: [1, 2],
#     # 3: [1],
#    4: [1, 2],
#     # 5: [1],
#     # 6: [1],
#     # 7: [1, 2],
#     # 8: [1, 2],
#     # 9: [1, 2],
# }

classes_to_subclasses = {
    1: [1, 3],  # OK
    2: [1, 2],  # OK
    3: [1],  # OK
    4: [1, 2], # OK
    6: [1],  # OK
    7: [1],  # OK
    9: [2],  # OK
}

In [ ]:
clean_kafka_topics()
# clean_neo4j_db()

for class_num in classes_to_subclasses:
    for subclass in classes_to_subclasses[class_num]:
        train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_post_process=True)

In [15]:
clean_kafka_topics()

# for class_number in classes_to_subclasses.keys():
#     generate_mnist_samples(
#         class_number, max_samples=1000, test_fraction=1, randomize=False
#     )

params = {
    # "feature_weight": 0.5,
    # "structural_weight": 0.5,
    "ged_timeout": 60,
    "skeletonization_threshold": 180,
    # "simplification_epsilon": 5,
}
test_mnist_all(classes_to_subclasses.keys(), params, sample_fraction=1)

Testing MNIST classification:   0%|          | 0/5962 [00:00<?, ?it/s]

/Users/mlapin/Development/personal/NaturalAGI/natural-agi/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mlapin/Development/personal/NaturalAGI/natural-agi/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


({'/opt/nuclio/shared_storage/generated_samples/mnist_1/test/mnist_1_00515.png': {'status': 'success',
   'image_path': '/opt/nuclio/shared_storage/generated_samples/mnist_1/test/mnist_1_00515.png',
   'classification_results': [{'concept_id': '1_3',
     'is_minor': True,
     'message': 'Similarity between image and concept 1_3: 0.7981',
     'similarity': 0.7981,
     'concept_complexity': 5,
     'image_complexity': 5}],
   'image_id': '982ebb86-df33-49b6-b57f-025fbfa53f05',
   'parameters': {'ged_timeout': 60,
    'image_path': '/opt/nuclio/shared_storage/generated_samples/mnist_1/test/mnist_1_00515.png',
    'skeletonization_threshold': 180,
    'image_id': '982ebb86-df33-49b6-b57f-025fbfa53f05',
    'session_id': 'fc155e3f-4acd-41ef-b299-ff0b864fa42d',
    'image_width': 100,
    'image_height': 100,
    'simplification_epsilon': 5.0},
   'profiling': {'skeletonization_time_ms': 213.9235,
    'contour_analysis_time_ms': 88.256292,
    'classification_time_ms': 248.580751},
   'e

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

delete_test_neo4j_nodes()

# Simple classification
class_number = 1
img_num = 448
image_id = f"mnist_{class_number}_{img_num:05d}"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
path = f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"


# Load and display the image
img_path = os.path.join(local_path, f"{image_id}.png")
if os.path.exists(img_path):
    img = mpimg.imread(img_path)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap='gray')
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis('off')
    plt.show()
else:
    print(f"Image not found at: {img_path}")


image_id_for_test = str(uuid.uuid4())
params = {
    # "feature_weight": 0.5,
    # "structural_weight": 0.5,
    "ged_timeout": 3,
    "concept_of_interest": f"{class_number}",
    "session_id": "test",
    "image_id": image_id_for_test,
    "skeletonization_threshold": 180,
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(path, f"{image_id}.png"), params=params, timeout=60)
print("\nFull classification result:")
print(json.dumps(result, indent=2))

if result["status"] == "success" and len(result["classification_results"]) > 0:
    activated_class = result["classification_results"][0]["concept_id"]
    print(f"Activated class: {activated_class}")
else:
    print(f"Classification failed: {result.get('error', 'Unknown error')}")

with open("../latest_results.json", "w") as f:
    json.dump(result, f)

In [ ]:
def remove_concept(concept_id: str) -> None:
    """Remove a concept from the database."""
    # Delete all nodes and relationships associated with the concept
    query = """
    MATCH (n)
    WHERE n.concept_id = $concept_id
    DETACH DELETE n
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        session.run(query, concept_id=concept_id)

In [ ]:
remove_concept("3_1")

In [ ]:
def retrain_concept(number: int, subclass: int, with_post_process: bool = True) -> None:
    """Retrain a concept."""
    remove_concept(f"{number}_{subclass}")
    train_mnist(class_number=number, subclass=subclass, is_prepared_samples=True, with_post_process=with_post_process)

In [ ]:
retrain_concept(number = 3, subclass = 1, with_post_process=False)